# Notebook 16 — C1.8: does the fingerprint's separability survive the C0 trees?

The C0 re-selection (notebook 15) scaled the two CNN trees up, and the *diagnostic* fingerprint silhouette fell (CIFAR 0.33 → 0.08, ImageNet 0.36 → 0.20). The MLPs — whose trees stayed small — did **not** drop, which suggests the fall is a **fingerprint-size artifact**: the wide `≥2` branching adds many deep, low-purity factors that dilute a cosine silhouette. This notebook tests that properly.

**Claim under test:** the BFT fingerprint is a *more class-separable* stimulus code than the network's own activations.

**Baselines** (each at **native** dimension and **dim-matched**):
- **penultimate activations** — the network's own last-layer representation (the classifier input);
- **full activations** — every traced layer's activations, spatially pooled and concatenated (the "it just spans many layers" control).

Dim-matching is done **per pair, to `min(dim_fp, dim_act)`**, by PCA *and* by Gaussian random projection, so neither code wins on raw dimensionality (this mirrors the paper's Fig P-d/-e). When the fingerprint is the larger of the two, matching reduces the *fingerprint* — the harder test for it.

**Two axes, run for the CNNs** (where the claim should be strongest):

1. **Model seeds.** Re-trace on each trained seed (CIFAR 0–4) and report mean ± SD across seeds — the honest error bar, not a stimulus bootstrap. SqueezeNet has one checkpoint, so it gets an **NMF-seed** sweep instead (`init='random'`, which is the only variance a single model admits; expect it small).
2. **Tree size / shape**, on seed 0:
   - `paper` — the original small HPs;
   - `c0` — the notebook-15 selection;
   - `narrow` — `c0` ranks, but branch only the output (no sub-splitting): thin, deep;
   - `wide` — `c0`, but widen the branching two layers before the output: bushy, shallow-heavy.

3. **Filtered slices of one fitted tree** (free — just column selection on an already-fitted tree). This isolates *where* the separability lives:
   - `output_only`, `top2`, `top_half` — **wide & shallow**, the class-specific factors near the output;
   - `spine` (follow only the top factor to the input), `bottom_half`, `input_only` — **deep & narrow**, the low-purity shared factors;
   - `full`, `no_output` — the whole tree and the whole tree minus the classifier.
   If the silhouette is carried by the shallow slices and *diluted* by the deep ones, the C0 drop is confirmed cosmetic and the fix is a slice / λ-weighted fingerprint, not a smaller tree.

**Metrics:** cosine **silhouette** and 3-fold CV **kNN accuracy**, for every representation, at native and matched dimension.

## Pipeline

| § | step |
|---|---|
| §2 | inherit nb09's pipeline; build each model's context at the **C0** HPs (from `nb15_hp_<exp>.json`) |
| §3 | representation + metric helpers (activations, fingerprint slices, PCA/GRP matching, sep) |
| §4 | **seed sweep** — trace per model-seed (or NMF-seed), score every rep + slice + paired-matched comparison |
| §5 | **tree-size sweep** — seed 0, the four tree shapes, same scoring |
| §6 | driver over `MODELS` |
| §7 | summary tables |

Checkpoints to `data/results/nb16_sep_<exp>.json` after **every** trace, so a killed run keeps what finished. Skips a model whose JSON is complete unless `NB16_FORCE=1`.

## §0 · Setup

In [ ]:
import os, sys, json, gc, time, warnings
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection

warnings.filterwarnings('ignore')
REPO   = os.path.abspath('..')
MODE   = os.environ.get('NB16_MODE', 'local')
FORCE  = os.environ.get('NB16_FORCE', '0') == '1'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))

FIG_DIR = os.path.join(REPO, 'figs', '16_separability')
RES_DIR = os.path.join(REPO, 'data', 'results')
HP_DIR  = os.path.join(REPO, 'figures', 'figdata')      # nb15 outputs live here
for d in (FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

if MODE == 'cluster':
    MODEL_SEEDS   = range(5)      # CNN / MLP: independent trained seeds
    NMF_SEEDS     = 5             # single-model archs (ImageNet, ViT): NMF re-seeds
    BFT_MAX_ITER  = 500
    N_TRACE       = None
    GRP_SEED      = 0
else:
    MODEL_SEEDS   = range(2)
    NMF_SEEDS     = 2
    BFT_MAX_ITER  = 120
    N_TRACE       = 400           # laptop cap (overrides nb09's N_TRACE in local mode)
    GRP_SEED      = 0

# bft() accepts only these keys as trace overrides; everything else in a C0 kwargs blob
# (conf_per_class, weighting labels, …) is stripped before it reaches bft.
_BFT_KEYS = {'k_max', 'n_branches', 'stimulus_threshold', 'weighting',
             'normalization', 'conv_pool_method'}


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    return o if isinstance(o, (float, int, str)) or o is None else str(o)


class Recorder:
    def __init__(self, exp):
        self.exp = exp
        self.path = os.path.join(RES_DIR, f'nb16_sep_{exp}.json')
        self.results = {'experiment': exp, 'notebook': '16', 'mode': MODE, 'complete': False}
        self.completed = []

    def checkpoint(self, section=None):
        if section and section not in self.completed:
            self.completed.append(section)
        self.results['completed_sections'] = list(self.completed)
        tmp = self.path + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(jsonable(self.results), f, indent=1)
        os.replace(tmp, self.path)
        print(f'  [checkpoint] {self.exp}:{section or ""} '
              f'({len(self.completed)} sections)')

    def done(self):
        self.results['complete'] = True
        self.checkpoint('DONE')


def savefig(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, bbox_inches='tight')
    print('  saved', os.path.relpath(p, REPO))
    plt.close(fig)
    return os.path.relpath(p, REPO)


print(f'MODE={MODE}  DEVICE={DEVICE}  MODEL_SEEDS={list(MODEL_SEEDS)}  '
      f'NMF_SEEDS={NMF_SEEDS}  FORCE={FORCE}')

## §1 · **Select the models to run here**

Default is the two CNNs — the focus of C1.8. Override with `NB16_MODELS=cnn_cifar` or edit the list. The C0 hyperparameters are read from `figures/figdata/nb15_hp_<exp>.json`; a model without that file falls back to its notebook-09 registry HPs (and prints a warning).

In [ ]:
MODELS = ['cnn_cifar', 'imagenet_cnn']
if os.environ.get('NB16_MODELS'):
    MODELS = [m.strip() for m in os.environ['NB16_MODELS'].split(',') if m.strip()]

# Which tree shapes to fit in the seed-0 tree-size sweep (§5).
TREE_CONFIGS = ['paper', 'narrow', 'c0', 'wide'] if MODE == 'cluster' else ['c0', 'paper']

print('models:', MODELS, '| tree configs:', TREE_CONFIGS)

## §2 · Inherit nb09's pipeline, build each model at its C0 HPs

Same mechanism as notebook 15: `exec` nb09 §0–§2 into a fresh per-model namespace. Before `build_experiment` runs, the registry's `bft` block is overwritten with the C0 selection and `conf_per_class` with the C0 scale-up, so `ctx['tree']` is the C0 tree and `ctx['bft_call'](**overrides)` re-traces the same model at any HPs. The original registry HPs are stashed first as the `paper` tree config.

In [ ]:
import nbformat

_NB09 = '09_validation_all_models.ipynb'
_nb09 = nbformat.read(_NB09, as_version=4)
_SETUP = []
for _c in _nb09.cells:
    if _c.cell_type != 'code':
        continue
    _SETUP.append(_c.source)
    if 'build_experiment(EXP)' in _c.source:
        break
print(f'inherited {len(_SETUP)} setup cells from {_NB09}')


def _c0_kwargs(exp):
    """C0 trace overrides + conf_per_class from nb15; None if the file is missing."""
    p = os.path.join(HP_DIR, f'nb15_hp_{exp}.json')
    if not os.path.exists(p):
        return None, None
    d = json.load(open(p))
    fbk = d.get('final_bft_kwargs', {})
    overrides = {k: v for k, v in fbk.items() if k in _BFT_KEYS}
    conf = (d.get('scale') or {}).get('conf_per_class')
    return overrides, conf


def load_ctx(exp):
    os.environ['NB09_EXP'], os.environ['NB09_MODE'] = exp, MODE
    ns = {'__name__': f'nb09_ctx_{exp}', '__builtins__': __builtins__}
    for i, src in enumerate(_SETUP[:-1]):                      # §0 + §1
        exec(compile(src, f'{_NB09}:setup{i}', 'exec'), ns)

    r = ns['REG'][exp]
    paper_bft = {k: v for k, v in dict(r['bft']).items() if k in _BFT_KEYS}
    c0, conf = _c0_kwargs(exp)
    if c0 is None:
        print(f'  WARNING {exp}: no nb15_hp_{exp}.json — using registry HPs as C0')
        c0 = paper_bft
    else:
        r['bft'] = dict(r['bft'])
        # build_experiment passes weighting= explicitly, so keep it (and normalization)
        # OUT of the registry blob to avoid a duplicate-kwarg collision.
        r['bft'].update({k: v for k, v in c0.items()
                         if k not in ('weighting', 'normalization')})
        if conf is not None and 'conf_per_class' in r:
            r['conf_per_class'] = conf
    ns['BFT_MAX_ITER'] = BFT_MAX_ITER
    ns['N_TRACE'] = None if MODE == 'cluster' else N_TRACE

    exec(compile(_SETUP[-1], f'{_NB09}:setup_build', 'exec'), ns)   # §2 build_experiment
    ns['ctx']['_paper_bft'] = paper_bft
    ns['ctx']['_c0_bft'] = c0
    ns['ctx']['_kind'] = r['kind']
    ns['ctx']['_conf_per_class'] = r.get('conf_per_class')
    return ns


def release(ns):
    ns.clear(); gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## §3 · Representations and metrics

- **Activations.** `penult` = the classifier's input (the network's own penultimate code); `full` = every traced layer's input, spatially pooled (conv: mean over H×W; attn: mean over tokens) and concatenated. Both are read off the *same* collected layer inputs the tree was built from, so they sit on identical stimuli in identical order.
- **Fingerprint slices.** One fitted tree → many fingerprints by selecting which nodes' loadings to concatenate. Depth and layer index are equivalent here (`depth = L_out − layer_idx`), so slices are defined by layer index and by the path (`spine` = the all-top-factor path).
- **Matching.** `match(X, d)` PCA- or GRP-reduces to `d`; a pair is compared at `d = min(dim_fp, dim_act)`.
- **`sep(X, y)`** returns cosine silhouette and 3-fold kNN accuracy, reusing nb09's `sep_metrics` (L2-normalizes rows, cosine silhouette, `knn_cv`).

In [ ]:
def pooled(a):
    a = np.asarray(a, dtype=np.float32)
    if a.ndim == 4:                       # (N,C,H,W) conv -> (N,C)
        return a.mean((2, 3))
    if a.ndim == 3:                       # (N,T,d) attn tokens -> (N,d)
        return a.mean(1)
    return a.reshape(len(a), -1)


def act_reps(layer_inputs):
    """penultimate and full-concatenation activation baselines."""
    layers = [pooled(li) for li in layer_inputs]
    return {'penult': layers[-1], 'full': np.concatenate(layers, axis=1)}


def _slice_predicates(L_out):
    """name -> predicate(layer_idx, path) selecting nodes for a fingerprint slice."""
    half = L_out // 2
    return {
        'full':          lambda li, p: True,
        'output_only':   lambda li, p: li == L_out,
        'top2':          lambda li, p: li >= L_out - 1,          # wide & shallow
        'top_half':      lambda li, p: li >= L_out - half,       # wide & shallow
        'no_output':     lambda li, p: li != L_out,
        'bottom_half':   lambda li, p: li <= half,               # near input
        'input_only':    lambda li, p: li == 0,
        'spine':         lambda li, p: all(int(x) == 0 for x in p),  # deep & narrow
    }


def fp_slice(tree, indices, keep):
    """Concatenate img_factors of the nodes for which keep(layer_idx, path) is True."""
    parts, queue = [], [tree.root if hasattr(tree, 'root') else tree]
    while queue:
        nd = queue.pop(0)
        if keep(int(nd.layer_idx), tuple(nd.path)):
            parts.append(np.asarray(nd.img_factors)[indices, :])
        queue.extend(nd.children)
    return np.concatenate(parts, axis=1) if parts else np.zeros((len(indices), 0))


def fp_all_slices(tree, n_rows, L_out):
    idx = np.arange(n_rows)
    preds = _slice_predicates(L_out)
    out = {name: fp_slice(tree, idx, pred) for name, pred in preds.items()}
    # per-layer slices, to see which single layer carries the silhouette
    for li in range(L_out + 1):
        out[f'L{li}'] = fp_slice(tree, idx, (lambda li_: (lambda l, p: l == li_))(li))
    return {k: v for k, v in out.items() if v.shape[1] > 0}


def match(X, d, how, seed=GRP_SEED):
    X = np.asarray(X, dtype=np.float32)
    d = int(min(d, X.shape[1], max(1, X.shape[0] - 1)))
    if d >= X.shape[1]:
        return X
    if how == 'pca':
        return PCA(n_components=d, random_state=0).fit_transform(X)
    return GaussianRandomProjection(n_components=d, random_state=seed).fit_transform(X)


def sep(ns, X, y):
    """cosine silhouette + 3-fold kNN accuracy via nb09's sep_metrics."""
    if X.shape[1] == 0:
        return {'sil': float('nan'), 'knn': float('nan'), 'dim': 0}
    s, k = ns['sep_metrics'](np.asarray(X, dtype=np.float32), y)
    return {'sil': float(s), 'knn': float(k), 'dim': int(X.shape[1])}


def paired_matched(ns, A, B, y):
    """A vs B at native, PCA-matched and GRP-matched to min(dim) — the fair comparison."""
    dm = int(min(A.shape[1], B.shape[1]))
    out = {'match_dim': dm,
           'A_native': sep(ns, A, y), 'B_native': sep(ns, B, y)}
    for how in ('pca', 'grp'):
        out[f'A_{how}'] = sep(ns, match(A, dm, how), y)
        out[f'B_{how}'] = sep(ns, match(B, dm, how), y)
    return out


def evaluate(ns, tree, y, layer_inputs):
    """Native sep of every rep + slice, plus the paired-matched headline comparisons."""
    y = np.asarray(y).astype(int)
    n_rows = tree.root.img_factors.shape[0]
    acts = act_reps(layer_inputs)
    slices = fp_all_slices(tree, n_rows, int(max(int(nd.layer_idx) for nd in tree.nodes())))

    native = {}
    for name, X in {**{f'act_{k}': v for k, v in acts.items()},
                    **{f'fp_{k}': v for k, v in slices.items()}}.items():
        native[name] = sep(ns, X, y)

    # headline paired comparisons (fingerprint vs each activation baseline), dim-matched
    fp_full = slices['full']
    pairs = {}
    for aname, A in acts.items():
        pairs[f'fp_full__vs__act_{aname}'] = paired_matched(ns, fp_full, A, y)
    # is the separability in the shallow slices? compare a wide-shallow and a deep-narrow
    for sname in ('output_only', 'top_half', 'spine', 'bottom_half'):
        if sname in slices:
            pairs[f'fp_{sname}__vs__act_penult'] = paired_matched(ns, slices[sname],
                                                                  acts['penult'], y)
    return {'n_rows': int(n_rows), 'fp_full_dim': int(fp_full.shape[1]),
            'penult_dim': int(acts['penult'].shape[1]), 'full_dim': int(acts['full'].shape[1]),
            'native': native, 'paired': pairs}

## §4 · Seed sweep

For a multi-seed architecture (CNN, MLP) each **trained seed** is an independent replicate: collect that seed's layer inputs on the C0 stimulus population, trace at the C0 HPs, score everything. For a single-model architecture (ImageNet SqueezeNet, ViT) the only available variance is the **NMF seed** (`init='random'`); the model, stimuli and activations are fixed, so only the fingerprint moves.

In [ ]:
def _collect_cfg(ns):
    """(loader, label_transform, fine_key) for collecting a seed's layer inputs.

    MLP: collect on the raw test loader with the parity transform, keep 'digits' as the
    fine label (the interesting one). CNN: collect on the confident subset; class == fine.
    """
    ctx = ns['ctx']; kind = ctx['_kind']
    if kind == 'mlp':
        return ctx['eval_loader'], ctx.get('label_transform'), 'digits'
    return ctx['vloader'], None, 'targets'


def seed_sweep(ns, rec):
    ctx = ns['ctx']; kind = ctx['_kind']
    over = {k: v for k, v in ctx['_c0_bft'].items() if k in _BFT_KEYS}
    reps, rows = {}, []

    if kind in ('mlp', 'cnn') and len(ctx.get('seeds', {})) >= 2:
        reps['replicate_type'] = 'model_seed'
        loader, ltf, fine_key = _collect_cfg(ns)
        for s, model_s in sorted(ctx['seeds'].items()):
            if s not in MODEL_SEEDS:
                continue
            t0 = time.perf_counter()
            raw = ns['collect_layer_dicts'](model_s, loader, only_correct=True,
                                            label_transform=ltf, device=ns['DEVICE'])
            y = raw[fine_key]
            layer_inputs = [d['input_fmap'] for d in raw['layer_data']]
            call_kw = {'weighting': 'img_selectivity', 'max_iter': BFT_MAX_ITER,
                       'n_jobs': 3}
            call_kw.update(over)          # C0 blob may carry weighting -> avoid double kwarg
            tree = ns['bft'](raw['layer_data'], **call_kw)
            rows.append({'seed': int(s), **evaluate(ns, tree, y, layer_inputs),
                         'wall_s': round(time.perf_counter() - t0, 1)})
            _log_row(rows[-1]); del tree, raw; gc.collect()
            rec.results['seed_sweep'] = {**reps, 'reps': rows}
            rec.checkpoint(f'seed{s}')
    else:
        reps['replicate_type'] = 'nmf_seed'
        y = ns['fine'].astype(int); layer_inputs = ns['layer_inputs']
        for s in range(NMF_SEEDS):
            t0 = time.perf_counter()
            tree = ctx['bft_call'](random_state=s, init='random', max_iter=BFT_MAX_ITER)
            rows.append({'seed': int(s), **evaluate(ns, tree, y, layer_inputs),
                         'wall_s': round(time.perf_counter() - t0, 1)})
            _log_row(rows[-1]); del tree; gc.collect()
            rec.results['seed_sweep'] = {**reps, 'reps': rows}
            rec.checkpoint(f'nmfseed{s}')
    return rec.results['seed_sweep']


def _log_row(r):
    n = r['native']
    def g(k): return n.get(k, {}).get('sil', float('nan'))
    print(f"    seed {r['seed']}: fp_full sil={g('fp_full'):.3f} knn={n['fp_full']['knn']:.3f}"
          f" (dim {r['fp_full_dim']}) | penult sil={g('act_penult'):.3f}"
          f" | full-act sil={g('act_full'):.3f}"
          f" | output-only sil={g('fp_output_only'):.3f} | spine sil={g('fp_spine'):.3f}"
          f"  ({r['wall_s']:.0f}s)")

## §5 · Tree-size sweep (seed 0)

Four tree shapes on the seed-0 model, same scoring. The activation baselines are identical across shapes (same model, same stimuli) — only the fingerprint changes — so this isolates the effect of tree size/shape on separability. `narrow` and `wide` are derived from the `c0` branch vector.

In [ ]:
def _tree_variants(ctx):
    """name -> bft override kwargs for the four shapes."""
    c0 = {k: v for k, v in ctx['_c0_bft'].items() if k in _BFT_KEYS}
    paper = {k: v for k, v in ctx['_paper_bft'].items() if k in _BFT_KEYS}
    km = list(c0.get('k_max', []))
    b = list(c0.get('n_branches', [1] * len(km)))
    L = len(b)
    out = {'c0': c0, 'paper': paper}
    if L >= 2:
        # narrow: branch only the output, no sub-splitting -> thin, deep
        nb_narrow = [1] * (L - 1) + [b[-1]]
        out['narrow'] = {**c0, 'n_branches': nb_narrow}
        # wide: widen the two layers before the output -> bushy, shallow-heavy
        nb_wide = list(b)
        nb_wide[-2] = min(km[-2] if km else 4, max(nb_wide[-2], 4))
        if L >= 3:
            nb_wide[-3] = min(km[-3] if km else 2, max(nb_wide[-3], 2))
        out['wide'] = {**c0, 'n_branches': nb_wide}
    return out


def tree_sweep(ns, rec):
    ctx = ns['ctx']
    y = ns['fine'].astype(int); layer_inputs = ns['layer_inputs']
    variants = _tree_variants(ctx)
    res = {}
    rec.results['tree_sweep'] = res
    for name in TREE_CONFIGS:
        if name not in variants:
            continue
        over = {k: v for k, v in variants[name].items() if k in _BFT_KEYS}
        t0 = time.perf_counter()
        tree = ctx['bft_call'](**over, max_iter=BFT_MAX_ITER)
        ev = evaluate(ns, tree, y, layer_inputs)
        res[name] = {'kwargs': jsonable(over), **ev,
                     'wall_s': round(time.perf_counter() - t0, 1)}
        n = ev['native']
        print(f"  tree '{name}': fp_full sil={n['fp_full']['sil']:.3f} "
              f"knn={n['fp_full']['knn']:.3f} (dim {ev['fp_full_dim']}, {ev['n_rows']} rows) "
              f"| penult sil={n['act_penult']['sil']:.3f}  ({res[name]['wall_s']:.0f}s)")
        del tree; gc.collect()
        rec.checkpoint(f'tree_{name}')
    return res

## §6 · Driver

In [ ]:
def agg_native(reps, keys=('sil', 'knn')):
    """mean/sd across replicates for every native representation."""
    names = sorted({k for r in reps for k in r['native']})
    out = {}
    for nm in names:
        for mk in keys:
            v = [r['native'][nm][mk] for r in reps
                 if nm in r['native'] and np.isfinite(r['native'][nm][mk])]
            if v:
                out.setdefault(nm, {})[mk] = {'mean': float(np.mean(v)),
                                              'sd': float(np.std(v)), 'n': len(v)}
    return out


SUMMARY = {}
for exp in MODELS:
    rec = Recorder(exp)
    if os.path.exists(rec.path) and not FORCE:
        try:
            prev = json.load(open(rec.path))
            if prev.get('complete'):
                print(f'\n=== {exp}: already complete — skipping (NB16_FORCE=1 to redo) ===')
                SUMMARY[exp] = prev; continue
        except Exception:
            pass

    print(f'\n{"=" * 74}\n=== {exp} ===\n{"=" * 74}')
    t_model = time.perf_counter()
    try:
        ns = load_ctx(exp)
    except Exception as e:
        print(f'  SKIPPED — {e}'); rec.results['error'] = str(e); rec.checkpoint('error'); continue

    rec.results.update(kind=ns['ctx']['_kind'], n_classes=int(ns['ctx']['n_classes']),
                       n_samples=int(ns['n_samples']),
                       c0_bft_kwargs=jsonable(ns['ctx']['_c0_bft']),
                       paper_bft_kwargs=jsonable(ns['ctx']['_paper_bft']))
    print(f'  n_samples={ns["n_samples"]}  n_classes={ns["ctx"]["n_classes"]}  '
          f'kind={ns["ctx"]["_kind"]}  C0={ns["ctx"]["_c0_bft"]}')
    rec.checkpoint('context')

    print('  -- seed sweep --')
    ss = seed_sweep(ns, rec)
    rec.results['seed_sweep']['agg_native'] = agg_native(ss['reps'])
    rec.checkpoint('seed_sweep')

    print('  -- tree-size sweep (seed 0) --')
    tree_sweep(ns, rec)
    rec.checkpoint('tree_sweep')

    rec.results['wall_s_total'] = round(time.perf_counter() - t_model, 1)
    rec.done()
    SUMMARY[exp] = rec.results
    release(ns)
    print(f'  {exp} done in {rec.results["wall_s_total"] / 60:.1f} min')

## §7 · Summary

Two questions, answered per model:

1. **Does the fingerprint beat the activation baselines, dim-matched?** — the paired comparisons, averaged over seeds.
2. **Where does the separability live?** — native silhouette of the slices; if `output_only`/`top_half` ≫ `spine`/`bottom_half`, the C0 drop is dilution by deep factors and the fix is a slice / λ-weighting, not a smaller tree.

In [ ]:
def _fmt(d): return f"{d['mean']:.3f}±{d['sd']:.3f}" if isinstance(d, dict) else f"{d:.3f}"

for exp in MODELS:
    r = SUMMARY.get(exp)
    if not r or 'seed_sweep' not in r:
        print(f'{exp}: (no result)'); continue
    ss = r['seed_sweep']; ag = ss.get('agg_native', {})
    print(f'\n{"=" * 74}\n{exp}  [{ss.get("replicate_type")}, n={len(ss["reps"])}]\n{"=" * 74}')

    print('  native silhouette / kNN (mean±sd over replicates):')
    order = ['fp_full', 'fp_output_only', 'fp_top_half', 'fp_no_output', 'fp_spine',
             'fp_bottom_half', 'fp_input_only', 'act_penult', 'act_full']
    for nm in order:
        if nm in ag:
            sil = _fmt(ag[nm].get('sil', {})); knn = _fmt(ag[nm].get('knn', {}))
            print(f'    {nm:18s} sil {sil:16s} knn {knn}')

    # headline dim-matched comparisons on the first replicate (matched dims are seed-stable)
    rep0 = ss['reps'][0]
    print('  dim-matched (seed0): fingerprint vs activation baseline')
    for pair, d in rep0['paired'].items():
        if not pair.startswith('fp_full__'):
            continue
        base = pair.split('__vs__')[1]
        print(f"    {base:12s} @dim {d['match_dim']:4d}:  "
              f"fp(pca) {d['A_pca']['sil']:.3f}/{d['A_pca']['knn']:.3f}  "
              f"act(pca) {d['B_pca']['sil']:.3f}/{d['B_pca']['knn']:.3f}   "
              f"fp(grp) {d['A_grp']['sil']:.3f}  act(grp) {d['B_grp']['sil']:.3f}   "
              f"(native fp {d['A_native']['sil']:.3f} vs act {d['B_native']['sil']:.3f})")

    if 'tree_sweep' in r and r['tree_sweep']:
        print('  tree-size sweep (seed 0): fp_full native sil / knn (dim), vs penult sil')
        for name, tr in r['tree_sweep'].items():
            n = tr['native']
            print(f"    {name:8s} fp {n['fp_full']['sil']:.3f}/{n['fp_full']['knn']:.3f} "
                  f"(dim {tr['fp_full_dim']:4d})   penult {n['act_penult']['sil']:.3f}")

out = os.path.join(RES_DIR, 'nb16_sep_summary.json')
with open(out, 'w') as f:
    json.dump(jsonable({'mode': MODE, 'models': MODELS,
                        'per_model': {e: SUMMARY[e] for e in SUMMARY}}), f, indent=1)
print('\nwrote', os.path.relpath(out, REPO))

## How to run on the cluster

```bash
cd notebooks

# both CNNs (the focus)
NB16_MODE=cluster \
  ../.venv/bin/python -m nbconvert --to notebook --execute \
  --output executed_16_separability.ipynb \
  --ExecutePreprocessor.timeout=400000 16_separability.ipynb

# or one model per job
NB16_MODE=cluster NB16_MODELS=cnn_cifar \
  ../.venv/bin/python -m nbconvert --to notebook --execute \
  --output executed_16_sep_cnn.ipynb \
  --ExecutePreprocessor.timeout=400000 16_separability.ipynb
```

| var | default | meaning |
|---|---|---|
| `NB16_MODE` | `local` | `cluster` = 5 model seeds, 5 NMF seeds, all four tree shapes, full S |
| `NB16_MODELS` | `cnn_cifar,imagenet_cnn` | comma-separated subset |
| `NB16_FORCE` | `0` | `1` re-runs a model whose JSON is already `complete` |

**Needs:** the C0 HP files `figures/figdata/nb15_hp_<exp>.json` (else it warns and falls back to registry HPs); `cifar10_cnn_seed{0..4}` for the CIFAR seed sweep (missing seeds are just skipped, fewer replicates); ImageNet val data for `imagenet_cnn`. A model with `<2` seed checkpoints automatically uses the NMF-seed sweep.

**Cost.** CIFAR: 5 seed traces + 4 tree traces at S≈2,000 ≈ 9 traces × ~1 h ≈ 9 h. ImageNet: 5 NMF traces + 4 tree traces at S=544 (273-node tree) ≈ 9 × ~2.7 h ≈ 24 h — run it as its own job. Metrics are cheap next to tracing. Checkpoints after every trace.

**Output.** `data/results/nb16_sep_<exp>.json` and `nb16_sep_summary.json`. Each carries, per replicate: native silhouette+kNN of every activation baseline and every fingerprint slice, and the dim-matched (PCA & GRP) paired comparisons.

**Reading the result — three things:**
1. `seed_sweep.agg_native.fp_full` vs `act_penult` / `act_full` — does the whole-tree fingerprint still win on average, and by how much once the error bar is a seed SD not a bootstrap?
2. the `paired` block — does it still win **dim-matched** (the honest comparison)?
3. the slice ladder — if `fp_output_only` / `fp_top_half` ≫ `fp_spine` / `fp_bottom_half`, the C0 silhouette drop is confirmed to be dilution by deep low-purity factors. **Then the fix is a slice or λ-weighted fingerprint** (keep the deep C0 tree for circuits/pruning), and the `tree_sweep` shows which shape recovers the separability without shrinking the circuit tree.